# 02 — Fine-tune ByT5 for Akkadian OCR Correction

Reads pre-generated pairs from `results/` and fine-tunes `google/byt5-small`.

**Output:** fine-tuned model saved to `results/byt5-akkadian/`

### How to run on Colab
1. Paste your GitHub repo URL into `GITHUB_REPO_URL` in the Setup cell and run it
2. Run the Upload cell — click "Choose Files" and select `synthetic_pairs.jsonl` and `ocr_pairs.jsonl`
3. Run remaining cells — training takes ~1 hour on L4, ~2-4 hours on T4

### How to run locally (NVIDIA GPU)
1. From the repo root: `jupyter notebook notebooks/02_finetune_byt5.ipynb`
2. Ensure `results/synthetic_pairs.jsonl` and `results/ocr_pairs.jsonl` exist
3. Run all cells

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import os, sys

ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_COLAB:
    # ← Paste Github Repo URL (if forked from original)
    GITHUB_REPO_URL = 'https://github.com/EvxLee/Byt5-Akkadian-OCR-Correction.git'
    REPO_DIR = '/content/' + GITHUB_REPO_URL.split('/')[-1].replace('.git', '')

    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {GITHUB_REPO_URL} {REPO_DIR}')

    os.chdir(REPO_DIR)
    os.makedirs('results', exist_ok=True)

    !pip install -q transformers datasets sacrebleu python-Levenshtein accelerate
else:
    os.chdir('..')

print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── Upload pair files (Colab only) ────────────────────────────────────────────
# Click "Choose Files" below and select both:
#   synthetic_pairs.jsonl  (from scripts/generate_synthetic_pairs.py)
#   ocr_pairs.jsonl        (from notebooks/01_boxes_ocr.ipynb)

if ON_COLAB:
    from google.colab import files
    from pathlib import Path

    uploaded = files.upload()
    for filename, content in uploaded.items():
        dest = Path('results') / filename
        dest.write_bytes(content)
        print(f'Saved → {dest}')
else:
    print('Local run — skipping upload, expecting files already in results/')

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

In [ ]:
import Levenshtein
import sacrebleu

def exact_match(predictions, references):
    correct = sum(p == r for p, r in zip(predictions, references))
    return correct / len(predictions)

def character_error_rate(predictions, references):
    scores = []
    for p, r in zip(predictions, references):
        if len(r) == 0:
            continue
        scores.append(Levenshtein.distance(p, r) / len(r))
    return sum(scores) / len(scores) if scores else 0.0

def chrf_score(predictions, references):
    return sacrebleu.corpus_chrf(predictions, [references], word_order=2).score

def bleu_score(predictions, references):
    return sacrebleu.corpus_bleu(predictions, [references]).score

def full_report(predictions, references, label='model'):
    return {
        'label': label,
        'exact_match': exact_match(predictions, references),
        'cer': character_error_rate(predictions, references),
        'chrf': chrf_score(predictions, references),
        'bleu': bleu_score(predictions, references),
    }

## 1. Load pre-generated pairs

`synthetic_pairs.jsonl` was generated by `scripts/generate_synthetic_pairs.py` and already
contains the train/val/test split. We just load and filter by the `split` field.

If `ocr_pairs.jsonl` exists (from the Tesseract notebook), those pairs are added to the
training set only — they don't have split labels so we don't use them for val/test.

In [ ]:
RESULTS_DIR = Path('results')
SYNTHETIC_PAIRS = RESULTS_DIR / 'synthetic_pairs.jsonl'
OCR_PAIRS       = RESULTS_DIR / 'ocr_pairs.jsonl'

assert SYNTHETIC_PAIRS.exists(), f'Missing {SYNTHETIC_PAIRS} — run scripts/generate_synthetic_pairs.py first'

# Load synthetic pairs, split by the pre-assigned split field
train_pairs, val_pairs, test_pairs = [], [], []

with open(SYNTHETIC_PAIRS, encoding='utf-8') as f:
    for line in f:
        rec = json.loads(line)
        pair = (rec['noisy'], rec['gold'])
        if   rec['split'] == 'train': train_pairs.append(pair)
        elif rec['split'] == 'val':   val_pairs.append(pair)
        elif rec['split'] == 'test':  test_pairs.append(pair)

print(f'Synthetic — train: {len(train_pairs)}, val: {len(val_pairs)}, test: {len(test_pairs)}')

# Merge Tesseract pairs into train only (if available)
if OCR_PAIRS.exists():
    ocr = [json.loads(l) for l in OCR_PAIRS.read_text().splitlines()]
    ocr_train = [(r['noisy'], r['gold']) for r in ocr]
    train_pairs.extend(ocr_train)
    print(f'Added {len(ocr_train)} Tesseract pairs → train total: {len(train_pairs)}')
else:
    print('No ocr_pairs.jsonl found — using synthetic pairs only')

# Save test pairs separately for the evaluation notebook
with open(RESULTS_DIR / 'test_pairs.jsonl', 'w', encoding='utf-8') as f:
    for noisy, gold in test_pairs:
        f.write(json.dumps({'noisy': noisy, 'gold': gold}, ensure_ascii=False) + '\n')
print(f'Saved test_pairs.jsonl ({len(test_pairs)} pairs)')

## 2. Tokenize

In [ ]:
MODEL_NAME = 'google/byt5-small'
MAX_LEN    = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_dataset(pairs):
    noisy_list = [p[0] for p in pairs]
    gold_list  = [p[1] for p in pairs]
    enc = tokenizer(
        noisy_list, text_target=gold_list,
        max_length=MAX_LEN, truncation=True, padding=False,
    )
    return Dataset.from_dict(enc)

ds = DatasetDict({
    'train':      make_dataset(train_pairs),
    'validation': make_dataset(val_pairs),
    'test':       make_dataset(test_pairs),
})
print(ds)

## 3. Train

On a free Colab T4 GPU (~15GB VRAM), 5 epochs over ~61k training pairs takes roughly 2-4 hours.
The final epoch checkpoint is saved and used for evaluation.

In [ ]:
import torch, glob

CHECKPOINT_DIR = str(RESULTS_DIR / 'byt5-akkadian')

model    = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
collator = DataCollatorForSeq2Seq(tokenizer, model=model, pad_to_multiple_of=8)

total_steps  = (len(ds['train']) // (8 * 2)) * 5
warmup_steps = total_steps // 10

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=50,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    processing_class=tokenizer,
    data_collator=collator,
)

trainer.train()

# Reload explicitly from the last saved checkpoint — avoids in-memory state issues
checkpoints = sorted(
    glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'),
    key=lambda x: int(x.rsplit('-', 1)[-1])
)
if checkpoints:
    last_ckpt = checkpoints[-1]
    print(f'\nReloading from {last_ckpt}')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = AutoModelForSeq2SeqLM.from_pretrained(last_ckpt).to(device)
    print('Reload done.')
else:
    print('No checkpoint found — using in-memory model')

In [ ]:
# ── Sanity check ──────────────────────────────────────────────────────────────
device = next(model.parameters()).device
model.eval()

print('Generation config:', model.generation_config)
print(f'EOS id: {tokenizer.eos_token_id}  PAD id: {tokenizer.pad_token_id}')
print()

ids  = torch.tensor([ds['test'][0]['input_ids']]).to(device)
mask = torch.tensor([ds['test'][0]['attention_mask']]).to(device)

with torch.no_grad():
    out = model.generate(ids, attention_mask=mask, max_new_tokens=64, num_beams=1)

raw = out[0].tolist()
print('Raw output token IDs (first 15):', raw[:15])
print('Noisy :', test_pairs[0][0])
print('Pred  :', tokenizer.decode(out[0], skip_special_tokens=True))
print('Gold  :', test_pairs[0][1])

if len(raw) <= 2:
    print('\nModel is generating EOS immediately — paste the raw token IDs above for debugging')

## 4. Run predictions on the test set

In [ ]:
import torch
from tqdm.auto import tqdm

PRED_BATCH = 64
device = next(model.parameters()).device
model.eval()

decoded_preds = []

with torch.no_grad():
    for i in tqdm(range(0, len(ds['test']), PRED_BATCH), desc='Predicting'):
        batch      = ds['test'][i : i + PRED_BATCH]
        input_ids  = [torch.tensor(x) for x in batch['input_ids']]
        attn_masks = [torch.tensor(x) for x in batch['attention_mask']]

        # pad to same length within batch
        input_ids  = torch.nn.utils.rnn.pad_sequence(input_ids,  batch_first=True, padding_value=tokenizer.pad_token_id).to(device)
        attn_masks = torch.nn.utils.rnn.pad_sequence(attn_masks, batch_first=True, padding_value=0).to(device)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attn_masks,
            max_new_tokens=128,
            num_beams=1,
        )
        decoded_preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))

with open(RESULTS_DIR / 'model_predictions.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(decoded_preds))

print(f'Saved {len(decoded_preds)} predictions to results/model_predictions.txt')

## 5. Evaluate — model vs baseline

The baseline is the raw noisy input scored against gold — i.e., "do nothing".
The model must beat the baseline to have learned anything useful.

In [ ]:
test_noisy = [p[0] for p in test_pairs]
test_gold  = [p[1] for p in test_pairs]

baseline = full_report(test_noisy,    test_gold, label='baseline (noisy input, no model)')
model_r  = full_report(decoded_preds, test_gold, label='byt5-small fine-tuned')

for report in [baseline, model_r]:
    print(f"\n=== {report['label']} ===")
    print(f"  Exact match : {report['exact_match']:.3f}")
    print(f"  CER         : {report['cer']:.3f}  (lower is better)")
    print(f"  chrF++      : {report['chrf']:.2f}")
    print(f"  BLEU        : {report['bleu']:.2f}")

# Show a few example corrections
print('\n── Sample corrections (10 random from test) ──')
indices = random.sample(range(len(test_gold)), min(10, len(test_gold)))
for i in indices:
    print(f'  noisy : {test_noisy[i]}')
    print(f'  model : {decoded_preds[i]}')
    print(f'  gold  : {test_gold[i]}')
    print()